# Full Real + Fake Deepfake Benchmark Analysis

Local VS Code notebook for the current full benchmark schema. It reads the saved per-model benchmark CSV, derives all threshold-dependent predictions from `fake_probability`, and never reruns model inference.

The notebook compares real vs fake, clean vs stress, original vs resized, aspect vs square, stress type/level, and threshold behavior.


In [ ]:
from pathlib import Path
import glob

PROJECT_ROOT = Path(r"/Users/subrat/Desktop/Deepfake")
OUTPUT_BASE_DIR = PROJECT_ROOT / "output"
DEFAULT_MODEL_ACRONYM = "cf_vit"

# Leave MAIN_CSV_PATH as None to automatically load the latest benchmark_*.csv
# from the newest model output directory. You may also set it to an absolute CSV path.
MAIN_CSV_PATH = None

# Used only when an existing threshold sweep CSV is available. If missing, sweeps
# are recomputed from stored fake_probability scores only.
THRESHOLD_SWEEP_DIR = OUTPUT_BASE_DIR / DEFAULT_MODEL_ACRONYM

# Chart PNGs and derived analysis CSVs are saved here.
OUTPUT_DIR = OUTPUT_BASE_DIR / DEFAULT_MODEL_ACRONYM / "analysis"

DEFAULT_THRESHOLD = 0.50
THRESHOLDS = [round(x / 100, 2) for x in range(10, 100, 5)]
EXPECTED_PROCESSING_VARIANTS = [
    "original",
    "1024_aspect", "1024_square",
    "720_aspect", "720_square",
    "512_aspect", "512_square",
    "384_aspect", "384_square",
    "256_aspect", "256_square",
]
EXPECTED_STRESS_TYPES = ["blur", "brightness", "sharpness", "contrast", "jpeg_compression"]
TRANSITION_ORDER = [
    "TN_to_TN", "TN_to_FP", "FP_to_TN", "FP_to_FP",
    "TP_to_TP", "TP_to_FN", "FN_to_TP", "FN_to_FN",
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def latest_benchmark_csv() -> Path:
    candidates = sorted(OUTPUT_BASE_DIR.glob("*/benchmark_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No benchmark_*.csv found under {OUTPUT_BASE_DIR}")
    return candidates[0]


def resolve_main_csv() -> Path:
    if MAIN_CSV_PATH is not None:
        path = Path(MAIN_CSV_PATH).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(path)
        return path
    return latest_benchmark_csv()

MAIN_CSV_PATH = resolve_main_csv()
MODEL_OUTPUT_DIR = MAIN_CSV_PATH.parent
THRESHOLD_SWEEP_DIR = MODEL_OUTPUT_DIR
OUTPUT_DIR = MODEL_OUTPUT_DIR / "analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Main CSV     : {MAIN_CSV_PATH}")
print(f"Sweep dir    : {THRESHOLD_SWEEP_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")


## Setup

This installs only local analysis packages if they are missing. It does not download data, load the model, or rerun inference.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "kaleido": "kaleido",
}

for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])


## Load And Validate Data

This cell loads the centralized benchmark CSV, removes error rows from scored analysis, validates labels and score ranges, checks saved `result` against threshold-derived results, and drops duplicate `variant_id` rows for analysis so variants are not counted twice.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelcolor": "#2f2f33",
    "text.color": "#2f2f33",
    "xtick.color": "#3d3d42",
    "ytick.color": "#3d3d42",
})

PALETTE = {
    "real": "#2678B2",
    "fake": "#D95F02",
    "clean": "#2A9D8F",
    "stress": "#E76F51",
    "threshold": "#B00020",
    "safe": "#2A9D8F",
    "risk": "#D62828",
    "neutral": "#7A7A7A",
    "purple": "#7B2CBF",
}

raw_df = pd.read_csv(MAIN_CSV_PATH, low_memory=False)
print(f"Loaded rows: {len(raw_df):,}")
print(f"Columns: {len(raw_df.columns):,}")

required_columns = [
    "variant_id", "parent_image_id", "test_type", "actual_label", "fake_probability",
    "prediction", "result", "processing_variant", "target_dimension", "resize_mode",
    "stress_type", "stress_level", "score_delta_vs_clean",
]
missing_columns = [col for col in required_columns if col not in raw_df.columns]
if missing_columns:
    raise ValueError(f"Missing required current-schema columns: {missing_columns}")

validation_notes = []
df = raw_df.copy()
for col in ["test_type", "actual_label", "prediction", "result", "processing_variant", "target_dimension", "resize_mode", "stress_type", "stress_level"]:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

bad_labels = sorted(set(df.loc[~df["actual_label"].isin(["real", "fake"]), "actual_label"].astype(str)))
if bad_labels:
    validation_notes.append(f"Invalid actual_label values: {bad_labels}")

score_numeric = pd.to_numeric(df["fake_probability"], errors="coerce")
bad_score_count = int(score_numeric.isna().sum() + ((score_numeric < 0) | (score_numeric > 1)).sum())
if bad_score_count:
    validation_notes.append(f"Rows with missing/out-of-range fake_probability: {bad_score_count:,}")
df["score"] = score_numeric

error_mask = df.get("error", pd.Series("", index=df.index)).fillna("").astype(str).str.strip().ne("")
if int(error_mask.sum()):
    validation_notes.append(f"Rows ignored because error is non-empty: {int(error_mask.sum()):,}")

scored = df[
    ~error_mask
    & df["actual_label"].isin(["real", "fake"])
    & df["score"].notna()
    & df["score"].between(0, 1)
].copy()

if scored["variant_id"].duplicated().any():
    duplicate_count = int(scored["variant_id"].duplicated().sum())
    validation_notes.append(f"Duplicate variant_id rows dropped for analysis: {duplicate_count:,}")
    scored = scored.drop_duplicates("variant_id", keep="last").copy()


def label_to_binary(label):
    return 1 if str(label).strip().lower() == "fake" else 0


def prediction_at_threshold(scores, threshold=DEFAULT_THRESHOLD):
    return np.where(pd.to_numeric(scores, errors="coerce") >= threshold, "fake", "real")


def result_from_labels(actual, predicted):
    actual = np.asarray(actual, dtype=object)
    predicted = np.asarray(predicted, dtype=object)
    return np.select(
        [
            (actual == "fake") & (predicted == "fake"),
            (actual == "real") & (predicted == "real"),
            (actual == "real") & (predicted == "fake"),
            (actual == "fake") & (predicted == "real"),
        ],
        ["TP", "TN", "FP", "FN"],
        default="INVALID",
    )

scored["derived_prediction"] = prediction_at_threshold(scored["score"], DEFAULT_THRESHOLD)
scored["derived_result"] = result_from_labels(scored["actual_label"], scored["derived_prediction"])

if "result" in scored.columns:
    inconsistent = scored[scored["result"].fillna("").astype(str).str.strip().ne(scored["derived_result"])]
    if len(inconsistent):
        validation_notes.append(f"Rows where saved result differs from threshold-derived result at {DEFAULT_THRESHOLD:.2f}: {len(inconsistent):,}")

scored["test_scope"] = np.where(scored["test_type"].eq("stress"), "stress", "clean")
clean_df = scored[scored["test_scope"].eq("clean")].copy()
stress_df = scored[scored["test_scope"].eq("stress")].copy()
combined_df = scored.copy()

print("Validation notes:")
if validation_notes:
    for note in validation_notes:
        print(f"- {note}")
else:
    print("- No validation problems found.")

print("\nUsable scored rows:")
print(scored["test_scope"].value_counts(dropna=False).to_string())
print("\nGround-truth labels:")
print(scored["actual_label"].value_counts(dropna=False).to_string())


## Metric Helpers

All classification metrics below are derived from `fake_probability` at the selected threshold. Undefined values are shown as `N/A`, not silently converted to zero.


In [ ]:
def safe_div(n, d):
    return np.nan if d == 0 else float(n / d)


def auc_trapezoid(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2:
        return np.nan
    if hasattr(np, "trapezoid"):
        return float(np.trapezoid(y, x))
    return float(np.trapz(y, x))


def roc_curve_manual(y_true, scores):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    if len(np.unique(y)) < 2:
        return np.array([]), np.array([]), np.nan
    order = np.argsort(-s, kind="mergesort")
    y_sorted = y[order]
    pos = max(int((y == 1).sum()), 1)
    neg = max(int((y == 0).sum()), 1)
    tpr = np.r_[0, np.cumsum(y_sorted == 1) / pos, 1]
    fpr = np.r_[0, np.cumsum(y_sorted == 0) / neg, 1]
    return fpr, tpr, auc_trapezoid(fpr, tpr)


def pr_curve_manual(y_true, scores):
    y = np.asarray(y_true, dtype=int)
    s = np.asarray(scores, dtype=float)
    if int((y == 1).sum()) == 0:
        return np.array([]), np.array([]), np.nan
    order = np.argsort(-s, kind="mergesort")
    y_sorted = y[order]
    tp = np.cumsum(y_sorted == 1)
    fp = np.cumsum(y_sorted == 0)
    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / max(int((y == 1).sum()), 1)
    precision = np.r_[1.0, precision]
    recall = np.r_[0.0, recall]
    return recall, precision, auc_trapezoid(recall, precision)


def metrics_for(frame, threshold=DEFAULT_THRESHOLD):
    valid = frame.copy()
    valid = valid[valid["actual_label"].isin(["real", "fake"]) & valid["score"].notna() & valid["score"].between(0, 1)]
    count = int(len(valid))
    if count == 0:
        return {
            "Total samples": 0, "Real samples": 0, "Fake samples": 0,
            "TP": 0, "TN": 0, "FP": 0, "FN": 0,
            "Accuracy": np.nan, "Balanced Accuracy": np.nan, "Precision": np.nan,
            "Recall / TPR / Sensitivity": np.nan, "Specificity / TNR": np.nan,
            "FPR": np.nan, "FNR": np.nan, "F1 Score": np.nan, "ROC-AUC": np.nan, "PR-AUC": np.nan,
        }
    pred = prediction_at_threshold(valid["score"], threshold)
    result = result_from_labels(valid["actual_label"], pred)
    tp = int((result == "TP").sum())
    tn = int((result == "TN").sum())
    fp = int((result == "FP").sum())
    fn = int((result == "FN").sum())
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    tnr = safe_div(tn, tn + fp)
    fpr = safe_div(fp, fp + tn)
    fnr = safe_div(fn, fn + tp)
    f1 = safe_div(2 * precision * recall, precision + recall) if not (np.isnan(precision) or np.isnan(recall)) else np.nan
    y_true = valid["actual_label"].map(label_to_binary).astype(int)
    _fpr_curve, _tpr_curve, roc_auc = roc_curve_manual(y_true, valid["score"])
    _recall_curve, _precision_curve, pr_auc = pr_curve_manual(y_true, valid["score"])
    return {
        "Total samples": count,
        "Real samples": int((valid["actual_label"] == "real").sum()),
        "Fake samples": int((valid["actual_label"] == "fake").sum()),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Accuracy": safe_div(tp + tn, count),
        "Balanced Accuracy": np.nanmean([recall, tnr]) if not (np.isnan(recall) and np.isnan(tnr)) else np.nan,
        "Precision": precision,
        "Recall / TPR / Sensitivity": recall,
        "Specificity / TNR": tnr,
        "FPR": fpr,
        "FNR": fnr,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
    }


def compact_metrics_for(frame, threshold=DEFAULT_THRESHOLD):
    m = metrics_for(frame, threshold)
    return {
        "count": m["Total samples"], "TP": m["TP"], "TN": m["TN"], "FP": m["FP"], "FN": m["FN"],
        "Accuracy": m["Accuracy"], "FPR": m["FPR"], "FNR": m["FNR"],
        "TPR": m["Recall / TPR / Sensitivity"], "TNR": m["Specificity / TNR"], "F1": m["F1 Score"],
    }


def format_metric_value(value):
    if pd.isna(value):
        return "N/A"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        return f"{value:.4f}"
    return str(value)


def dataframe_elementwise_map(frame, func):
    mapped = frame.copy()
    for column in mapped.columns:
        mapped[column] = mapped[column].map(func)
    return mapped


def metrics_table(rows):
    out = pd.DataFrame(rows).set_index("Scope")
    return dataframe_elementwise_map(out, format_metric_value)


def confusion_matrix_counts(frame, threshold=DEFAULT_THRESHOLD):
    pred = prediction_at_threshold(frame["score"], threshold)
    matrix = pd.DataFrame(0, index=["real", "fake"], columns=["real", "fake"])
    counts = pd.crosstab(frame["actual_label"], pred)
    for actual in matrix.index:
        for predicted in matrix.columns:
            if actual in counts.index and predicted in counts.columns:
                matrix.loc[actual, predicted] = int(counts.loc[actual, predicted])
    return matrix


## Complete Benchmark Metrics

These tables summarize clean, stress, and combined rows independently. The confusion matrices use the default threshold and are derived from `fake_probability`, not from old real-only score columns.


In [ ]:
metric_rows = []
for scope_name, frame in [("Overall clean", clean_df), ("Overall stress", stress_df), ("Combined clean + stress", combined_df)]:
    metric_rows.append({"Scope": scope_name, **metrics_for(frame, DEFAULT_THRESHOLD)})

summary_metrics = pd.DataFrame(metric_rows)
display(metrics_table(metric_rows))
summary_metrics.to_csv(OUTPUT_DIR / "benchmark_scope_metrics.csv", index=False)

print("Clean confusion matrix")
display(confusion_matrix_counts(clean_df, DEFAULT_THRESHOLD))
print("\nStress confusion matrix")
display(confusion_matrix_counts(stress_df, DEFAULT_THRESHOLD))
print("\nCombined confusion matrix")
display(confusion_matrix_counts(combined_df, DEFAULT_THRESHOLD))


## Processing-Level Metrics

Clean rows are grouped by `processing_variant`, `target_dimension`, and `resize_mode` so original, aspect-preserving resize, and square crop behavior can be compared directly.


In [ ]:
def grouped_metrics(frame, group_cols, include_delta=False):
    rows = []
    if frame.empty:
        return pd.DataFrame(columns=group_cols)
    for keys, group in frame.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update(compact_metrics_for(group, DEFAULT_THRESHOLD))
        if include_delta:
            delta = pd.to_numeric(group.get("score_delta_vs_clean", pd.Series(dtype=float)), errors="coerce")
            row["mean score_delta_vs_clean"] = delta.mean()
            row["median score_delta_vs_clean"] = delta.median()
        rows.append(row)
    return pd.DataFrame(rows)

processing_metrics = grouped_metrics(clean_df, ["processing_variant", "target_dimension", "resize_mode"])
if not processing_metrics.empty:
    order_map = {name: i for i, name in enumerate(EXPECTED_PROCESSING_VARIANTS)}
    processing_metrics["_order"] = processing_metrics["processing_variant"].map(order_map).fillna(999)
    processing_metrics = processing_metrics.sort_values(["_order", "processing_variant"]).drop(columns="_order")

display(processing_metrics)
processing_metrics.to_csv(OUTPUT_DIR / "processing_level_metrics.csv", index=False)


## Stress-Level Metrics

Stress rows are grouped by stress family, severity, processing variant, target dimension, and resize mode. Delta summaries show how much the stress score moved relative to its matched clean baseline.


In [ ]:
stress_level_metrics = grouped_metrics(
    stress_df,
    ["stress_type", "stress_level", "processing_variant", "target_dimension", "resize_mode"],
    include_delta=True,
)
display(stress_level_metrics)
stress_level_metrics.to_csv(OUTPUT_DIR / "stress_level_metrics.csv", index=False)


## Clean-To-Stress Transition Analysis

For each stress row, the notebook uses the matched clean baseline fields already stored in the CSV. Counts and percentages are both reported; percentages are preferred for visual comparison.


In [ ]:
def transition_table(frame):
    if frame.empty or "prediction_transition" not in frame.columns:
        return pd.DataFrame(columns=["prediction_transition", "count", "percentage"])
    transitions = frame["prediction_transition"].fillna("").astype(str)
    transitions = transitions[transitions.isin(TRANSITION_ORDER)]
    counts = transitions.value_counts().reindex(TRANSITION_ORDER, fill_value=0)
    total = int(counts.sum())
    out = pd.DataFrame({
        "prediction_transition": counts.index,
        "count": counts.values,
        "percentage": [(value / total * 100) if total else np.nan for value in counts.values],
    })
    return out

transitions_overall = transition_table(stress_df)
display(transitions_overall)
transitions_overall.to_csv(OUTPUT_DIR / "clean_to_stress_transitions.csv", index=False)

transition_by_stress = []
for stress_type, group in stress_df.groupby("stress_type", dropna=False):
    tbl = transition_table(group)
    tbl.insert(0, "stress_type", stress_type)
    transition_by_stress.append(tbl)
transition_by_stress = pd.concat(transition_by_stress, ignore_index=True) if transition_by_stress else pd.DataFrame()
transition_by_stress.to_csv(OUTPUT_DIR / "clean_to_stress_transitions_by_stress_type.csv", index=False)


## Threshold Sweep

Threshold-dependent metrics are recomputed from stored `fake_probability` only. ROC-AUC and PR-AUC are probability-ranking metrics and are not treated as threshold-dependent values.


In [ ]:
def threshold_sweep_for(frame, scope):
    rows = []
    for threshold in THRESHOLDS:
        m = metrics_for(frame, threshold)
        rows.append({
            "scope": scope,
            "threshold": threshold,
            "TP": m["TP"], "TN": m["TN"], "FP": m["FP"], "FN": m["FN"],
            "Accuracy": m["Accuracy"], "Balanced Accuracy": m["Balanced Accuracy"],
            "Precision": m["Precision"], "Recall / TPR": m["Recall / TPR / Sensitivity"],
            "TNR": m["Specificity / TNR"], "FPR": m["FPR"], "FNR": m["FNR"], "F1": m["F1 Score"],
        })
    return pd.DataFrame(rows)

threshold_sweep = pd.concat([
    threshold_sweep_for(clean_df, "clean"),
    threshold_sweep_for(stress_df, "stress"),
    threshold_sweep_for(combined_df, "combined"),
], ignore_index=True)

display(threshold_sweep.head(12))
threshold_sweep.to_csv(OUTPUT_DIR / "threshold_sweep_recomputed.csv", index=False)


## Plot Helpers

Heatmaps leave missing cells blank, show real zero values clearly, and use dynamic text color for readable count/percentage labels.


In [ ]:
def savefig(name):
    path = OUTPUT_DIR / f"{name}.png"
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")


def chart_note(text):
    display(Markdown(f"**This plot represents:** {text}"))


def annotate_heatmap(ax, data, labels=None, fmt=".1f", suffix="%"):
    values = data.to_numpy(dtype=float)
    if labels is None:
        labels = data.copy()
    finite = values[np.isfinite(values)]
    vmin = float(finite.min()) if finite.size else 0.0
    vmax = float(finite.max()) if finite.size else 1.0
    midpoint = (vmin + vmax) / 2
    for y in range(data.shape[0]):
        for x in range(data.shape[1]):
            value = values[y, x]
            if not np.isfinite(value):
                continue
            text_color = "white" if value > midpoint else "#1f1f1f"
            label_value = labels.iloc[y, x]
            if isinstance(label_value, str):
                text = label_value
            else:
                text = f"{label_value:{fmt}}{suffix}"
            if text == "":
                continue
            ax.text(x + 0.5, y + 0.5, text, ha="center", va="center", fontsize=8, color=text_color)


def plot_confusion_heatmap(frame, title, filename):
    matrix = confusion_matrix_counts(frame, DEFAULT_THRESHOLD)
    total = matrix.values.sum()
    percent = matrix / total * 100 if total else matrix.replace(0, np.nan)
    if total:
        labels = matrix.astype(str) + "\n" + percent.round(1).astype(str) + "%"
    else:
        labels = pd.DataFrame("", index=matrix.index, columns=matrix.columns)
    plt.figure(figsize=(5.5, 4.5))
    ax = sns.heatmap(percent, annot=False, cmap="YlOrRd", vmin=0, vmax=100, cbar_kws={"label": "% of rows"}, linewidths=0.6, linecolor="white")
    annotate_heatmap(ax, percent, labels=labels)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    savefig(filename)
    plt.show()


def no_samples_plot(title, filename):
    plt.figure(figsize=(7, 2.5))
    plt.axis("off")
    plt.title(title)
    savefig(filename)
    plt.show()


def metric_label(value):
    return "N/A" if pd.isna(value) else f"{value:.3f}"


## Chart 1: Clean Confusion Matrix Heatmap

Shows how clean variants split into TP, TN, FP, and FN at the default threshold. This is the baseline model behavior before controlled stress transformations.


In [ ]:
if clean_df.empty:
    no_samples_plot("Clean Confusion Matrix", "01_clean_confusion_matrix")
else:
    plot_confusion_heatmap(clean_df, "Clean Confusion Matrix", "01_clean_confusion_matrix")
chart_note("Clean rows only. Each cell shows count and percentage of clean samples at the default threshold.")


## Chart 2: Stress Confusion Matrix Heatmap

Shows confusion outcomes after one controlled stress factor is applied. Compare this with the clean matrix to see whether transformations increase false positives or false negatives.


In [ ]:
if stress_df.empty:
    no_samples_plot("Stress Confusion Matrix", "02_stress_confusion_matrix")
else:
    plot_confusion_heatmap(stress_df, "Stress Confusion Matrix", "02_stress_confusion_matrix")
chart_note("Stress rows only. Each cell shows count and percentage after one controlled stress transformation.")


## Chart 3: Real vs Fake Fake-Probability Distribution

Shows score separation between real and fake samples, split by clean/stress rows. Good separation means real scores stay low while fake scores stay high; overlap around the threshold indicates fragile decisions.


In [ ]:
if scored.empty:
    no_samples_plot("Real vs Fake Fake-Probability Distribution", "03_real_fake_score_distribution")
else:
    plt.figure(figsize=(10, 5.5))
    ax = sns.violinplot(
        data=scored,
        x="actual_label", y="score", hue="test_scope",
        order=["real", "fake"], hue_order=["clean", "stress"],
        palette={"clean": PALETTE["clean"], "stress": PALETTE["stress"]},
        inner="quartile", cut=0, linewidth=0.8,
    )
    means = scored.groupby(["actual_label", "test_scope"])["score"].mean().reset_index()
    sns.stripplot(
        data=means, x="actual_label", y="score", hue="test_scope",
        order=["real", "fake"], hue_order=["clean", "stress"],
        dodge=True, marker="D", size=7, edgecolor="black", linewidth=0.7,
        palette={"clean": "#0B6E69", "stress": "#A33D2E"}, ax=ax,
    )
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], title="Row type", frameon=False, loc="upper center", ncol=2)
    ax.axhline(DEFAULT_THRESHOLD, color=PALETTE["threshold"], linestyle="--", linewidth=1.6)
    counts = scored.groupby(["actual_label", "test_scope"]).size().reset_index(name="n")
    for _, row in counts.iterrows():
        xpos = {"real": 0, "fake": 1}.get(row["actual_label"], 0)
        offset = -0.18 if row["test_scope"] == "clean" else 0.18
        ax.text(xpos + offset, 1.015, f"n={int(row['n']):,}", ha="center", va="bottom", fontsize=8, color="#444444")
    ax.set_ylim(-0.02, 1.08)
    ax.set_title("Real vs Fake Fake-Probability Distribution")
    ax.set_xlabel("Actual label")
    ax.set_ylabel("Fake probability")
    savefig("03_real_fake_score_distribution")
    plt.show()
chart_note("Score distributions by true class and row type. The dashed line is the default decision threshold; sample counts are shown above each group.")


## Chart 4: Overall Threshold Sweep

Shows how decision metrics move as the threshold changes. FPR and FNR expose failure rates, TPR and precision show fake-detection behavior, F1 balances precision/recall, and balanced accuracy averages real/fake class performance.


In [ ]:
if threshold_sweep.empty:
    no_samples_plot("Overall Threshold Sweep", "04_overall_threshold_sweep")
else:
    plot_df = threshold_sweep[threshold_sweep["scope"].eq("combined")].copy()
    metric_styles = {
        "FPR": (PALETTE["risk"], "--"),
        "Recall / TPR": (PALETTE["safe"], "-"),
        "FNR": (PALETTE["purple"], "--"),
        "Precision": ("#F4A261", "-"),
        "F1": ("#264653", "-"),
        "Balanced Accuracy": ("#457B9D", "-"),
    }
    plt.figure(figsize=(10.5, 5.8))
    for metric, (color, linestyle) in metric_styles.items():
        plt.plot(plot_df["threshold"], plot_df[metric], color=color, linestyle=linestyle, marker="o", markersize=3, linewidth=1.7, label=metric)
    plt.axvline(DEFAULT_THRESHOLD, color=PALETTE["threshold"], linestyle=":", linewidth=1.6, label=f"default {DEFAULT_THRESHOLD:.2f}")
    plt.ylim(-0.02, 1.02)
    plt.title("Overall Threshold Sweep - Combined Clean + Stress")
    plt.xlabel("Decision threshold")
    plt.ylabel("Metric value")
    plt.legend(frameon=False, ncol=3, fontsize=8)
    savefig("04_overall_threshold_sweep")
    plt.show()
chart_note("Combined clean+stress rows. Curves show decision metrics recalculated from stored fake_probability at each threshold; no inference is rerun.")


## Chart 5: Clean vs Stress Metrics Comparison

Compares key operating metrics at the default threshold. Look for stress rows lowering accuracy, F1, TPR, or TNR, or raising FPR/FNR.


In [ ]:
comparison_rows = []
for scope_name, frame in [("clean", clean_df), ("stress", stress_df)]:
    m = metrics_for(frame, DEFAULT_THRESHOLD)
    n = m["Total samples"]
    comparison_rows.extend([
        {"scope": scope_name, "metric": "Accuracy", "value": m["Accuracy"], "n": n},
        {"scope": scope_name, "metric": "F1", "value": m["F1 Score"], "n": n},
        {"scope": scope_name, "metric": "TPR", "value": m["Recall / TPR / Sensitivity"], "n": n},
        {"scope": scope_name, "metric": "TNR", "value": m["Specificity / TNR"], "n": n},
        {"scope": scope_name, "metric": "FPR", "value": m["FPR"], "n": n},
        {"scope": scope_name, "metric": "FNR", "value": m["FNR"], "n": n},
    ])
comparison_df = pd.DataFrame(comparison_rows)
if comparison_df["value"].notna().sum() == 0:
    no_samples_plot("Clean vs Stress Metrics Comparison", "05_clean_vs_stress_metrics")
else:
    plt.figure(figsize=(9.5, 5.2))
    ax = sns.barplot(data=comparison_df, x="metric", y="value", hue="scope", palette={"clean": PALETTE["clean"], "stress": PALETTE["stress"]})
    ax.set_ylim(0, 1)
    ax.set_title("Clean vs Stress Metrics Comparison")
    ax.set_xlabel("")
    ax.set_ylabel("Rate")
    ax.legend(title="Row type", frameon=False)
    for patch, (_, row) in zip(ax.patches, comparison_df.iterrows()):
        if pd.notna(row["value"]):
            ax.text(patch.get_x() + patch.get_width()/2, patch.get_height() + 0.015, f"{row['value']:.2f}\nn={int(row['n']):,}", ha="center", va="bottom", fontsize=7)
    savefig("05_clean_vs_stress_metrics")
    plt.show()
chart_note("Default-threshold comparison of clean and stress rows. Labels show metric value and row count for each bar.")


## Chart 6: Processing Variant Performance

FPR and FNR are split into separate subplots so real-sample false alarms and fake-sample misses can be inspected independently. Zero values are labeled explicitly.


In [ ]:
if processing_metrics.empty:
    no_samples_plot("Processing Variant Performance", "06_processing_variant_performance")
else:
    plot_df = processing_metrics.copy()
    for col in ["FPR", "FNR", "count"]:
        plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce")
    plot_df = plot_df.set_index("processing_variant").reindex(EXPECTED_PROCESSING_VARIANTS).reset_index()
    fig, axes = plt.subplots(2, 1, figsize=(11.5, 8.2), sharex=True)
    for ax, metric, color, title in [
        (axes[0], "FPR", PALETTE["risk"], "False Positive Rate by Processing Variant"),
        (axes[1], "FNR", PALETTE["purple"], "False Negative Rate by Processing Variant"),
    ]:
        sns.barplot(data=plot_df, x="processing_variant", y=metric, color=color, ax=ax)
        max_value = pd.to_numeric(plot_df[metric], errors="coerce").max()
        upper = 0.10 if pd.notna(max_value) and max_value <= 0.05 else 1.0
        ax.set_ylim(0, upper)
        ax.set_title(title)
        ax.set_ylabel(metric)
        ax.set_xlabel("")
        for patch, (_, row) in zip(ax.patches, plot_df.iterrows()):
            value = row.get(metric)
            count = row.get("count")
            if pd.isna(value):
                continue
            label = f"{value:.2f}\nn={int(count):,}" if pd.notna(count) else f"{value:.2f}"
            y = max(float(value), upper * 0.08)
            ax.text(patch.get_x() + patch.get_width()/2, y + upper * 0.025, label, ha="center", va="bottom", fontsize=7, color="#222222")
    axes[1].set_xlabel("Processing variant")
    axes[1].tick_params(axis="x", rotation=35)
    savefig("06_processing_variant_performance")
    plt.show()
chart_note("Clean rows grouped by processing variant. The top subplot shows FPR on real samples; the bottom shows FNR on fake samples, with counts on every bar.")


## Chart 7: Stress Type × Stress Level Performance Heatmap

Uses FPR for real samples and FNR for fake samples. Missing stress combinations are left blank; populated cells show sample count and percentage.


In [ ]:
def stress_failure_heatmap(label_filter, metric_name):
    frame = stress_df[stress_df["actual_label"].eq(label_filter)].copy()
    rows = []
    for keys, group in frame.groupby(["stress_type", "stress_level"], dropna=False):
        m = metrics_for(group, DEFAULT_THRESHOLD)
        rows.append({"stress_type": keys[0], "stress_level": keys[1], metric_name: m[metric_name], "count": m["Total samples"]})
    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(index=EXPECTED_STRESS_TYPES), pd.DataFrame(index=EXPECTED_STRESS_TYPES)
    rate = out.pivot(index="stress_type", columns="stress_level", values=metric_name).reindex(EXPECTED_STRESS_TYPES)
    counts = out.pivot(index="stress_type", columns="stress_level", values="count").reindex(EXPECTED_STRESS_TYPES)
    return rate, counts

real_fpr, real_counts = stress_failure_heatmap("real", "FPR")
fake_fnr, fake_counts = stress_failure_heatmap("fake", "FNR")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.4), sharey=False)
for ax, data, counts, title in [(axes[0], real_fpr, real_counts, "Real Samples: FPR"), (axes[1], fake_fnr, fake_counts, "Fake Samples: FNR")]:
    if data.empty or data.shape[1] == 0:
        ax.axis("off")
        ax.set_title(title)
    else:
        percent = data * 100
        sns.heatmap(percent, annot=False, cmap="YlOrRd", vmin=0, vmax=100, linewidths=0.5, linecolor="white", cbar_kws={"label": "%"}, ax=ax)
        labels = percent.copy().astype(object)
        for row_label in percent.index:
            for col_label in percent.columns:
                value = percent.loc[row_label, col_label]
                n = counts.loc[row_label, col_label] if row_label in counts.index and col_label in counts.columns else np.nan
                labels.loc[row_label, col_label] = "" if pd.isna(value) else f"n={int(n):,}\n{value:.1f}%"
        annotate_heatmap(ax, percent, labels=labels)
        ax.set_title(title)
        ax.set_xlabel("Stress level")
        ax.set_ylabel("Stress type")
plt.suptitle("Stress Type × Stress Level Performance", y=1.02, fontweight="bold")
savefig("07_stress_type_level_performance")
plt.show()
chart_note("Stress rows grouped by stress type and level. Cells show sample count and the relevant failure rate: FPR for real rows, FNR for fake rows.")


## Chart 8: Clean-To-Stress Transition Percentages

Shows how often clean outcomes stay stable or flip after stress. Percentages prevent sources or stress groups with different sample counts from dominating the visual.


In [ ]:
if transitions_overall.empty or transitions_overall["count"].sum() == 0:
    no_samples_plot("Clean-to-Stress Transition Percentages", "08_clean_to_stress_transition_percentages")
else:
    plot_df = transitions_overall.copy()
    plt.figure(figsize=(10, 4.8))
    colors = [PALETTE["safe"] if "_to_TN" in t or "_to_TP" in t else PALETTE["risk"] for t in plot_df["prediction_transition"]]
    ax = sns.barplot(data=plot_df, x="prediction_transition", y="percentage", palette=colors)
    ax.set_ylim(0, max(100, plot_df["percentage"].max() * 1.15))
    ax.set_title("Clean-to-Stress Transition Percentages")
    ax.set_xlabel("Transition")
    ax.set_ylabel("Percentage of stress rows")
    ax.tick_params(axis="x", rotation=35)
    for patch, (_, row) in zip(ax.patches, plot_df.iterrows()):
        value = row["percentage"]
        count = row["count"]
        if pd.notna(value):
            y = max(float(value), 0.8)
            ax.text(patch.get_x() + patch.get_width()/2, y + 1, f"n={int(count):,}\n{value:.1f}%", ha="center", va="bottom", fontsize=8)
    savefig("08_clean_to_stress_transition_percentages")
    plt.show()
chart_note("Stress rows linked to their matched clean baseline. Each bar shows the percentage and count for that clean-to-stress outcome transition.")


## Chart 9: Score Delta vs Clean Distribution By Stress Type

Shows how much each stress family shifts fake probability relative to the matched clean baseline. Positive deltas push rows toward fake; negative deltas push rows toward real.


In [ ]:
delta_df = stress_df.copy()
delta_df["score_delta_vs_clean"] = pd.to_numeric(delta_df.get("score_delta_vs_clean"), errors="coerce")
delta_df = delta_df[delta_df["score_delta_vs_clean"].notna()]
if delta_df.empty:
    no_samples_plot("Score Delta vs Clean by Stress Type", "09_score_delta_vs_clean_distribution")
else:
    plt.figure(figsize=(10, 5.4))
    ax = sns.boxplot(data=delta_df, x="stress_type", y="score_delta_vs_clean", order=EXPECTED_STRESS_TYPES, color="#A8DADC", showfliers=False)
    sns.stripplot(data=delta_df.sample(min(len(delta_df), 1200), random_state=42), x="stress_type", y="score_delta_vs_clean", order=EXPECTED_STRESS_TYPES, color="#264653", alpha=0.22, size=2, ax=ax)
    counts = delta_df.groupby("stress_type").size().reindex(EXPECTED_STRESS_TYPES)
    ymax = delta_df["score_delta_vs_clean"].max()
    for i, stress_type in enumerate(EXPECTED_STRESS_TYPES):
        n = counts.get(stress_type, np.nan)
        if pd.notna(n):
            ax.text(i, ymax + 0.03, f"n={int(n):,}", ha="center", va="bottom", fontsize=8)
    ax.axhline(0, color=PALETTE["neutral"], linestyle="--", linewidth=1.2)
    ax.set_ylim(delta_df["score_delta_vs_clean"].min() - 0.05, ymax + 0.12)
    ax.set_title("Score Delta vs Clean Distribution by Stress Type")
    ax.set_xlabel("Stress type")
    ax.set_ylabel("Fake probability delta")
    ax.tick_params(axis="x", rotation=25)
    savefig("09_score_delta_vs_clean_distribution")
    plt.show()
chart_note("Stress rows only. The y-axis is stress fake_probability minus the matched clean baseline score; counts show rows per stress type.")


## Chart 10: ROC Curves

ROC curves compare ranking quality for clean and stress rows. Curves nearer the top-left are better; AUC is undefined if a subset lacks either real or fake samples.


In [ ]:
plt.figure(figsize=(7.2, 6))
plotted = False
for scope, frame, color in [("clean", clean_df, PALETTE["clean"]), ("stress", stress_df, PALETTE["stress"] )]:
    if frame.empty:
        continue
    y = frame["actual_label"].map(label_to_binary).astype(int)
    fpr, tpr, auc_value = roc_curve_manual(y, frame["score"])
    if len(fpr):
        plt.plot(fpr, tpr, color=color, linewidth=2, label=f"{scope} AUC={metric_label(auc_value)} n={len(frame):,}")
        plotted = True
if plotted:
    plt.plot([0, 1], [0, 1], color="#999999", linestyle="--", linewidth=1)
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.title("ROC Curves")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend(frameon=False)
    savefig("10_roc_curves")
    plt.show()
else:
    no_samples_plot("ROC Curves", "10_roc_curves")
chart_note("Clean and stress score-ranking quality. Legend includes AUC and sample count for each curve.")


## Chart 11: Precision-Recall Curves

Precision-recall curves focus on fake detection quality. They are useful when real/fake counts are imbalanced or when fake recall is the main operating concern.


In [ ]:
plt.figure(figsize=(7.2, 6))
plotted = False
for scope, frame, color in [("clean", clean_df, PALETTE["clean"]), ("stress", stress_df, PALETTE["stress"] )]:
    if frame.empty:
        continue
    y = frame["actual_label"].map(label_to_binary).astype(int)
    recall, precision, auc_value = pr_curve_manual(y, frame["score"])
    if len(recall):
        plt.plot(recall, precision, color=color, linewidth=2, label=f"{scope} AUC={metric_label(auc_value)} n={len(frame):,}")
        plotted = True
if plotted:
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.title("Precision-Recall Curves")
    plt.xlabel("Recall / TPR")
    plt.ylabel("Precision")
    plt.legend(frameon=False)
    savefig("11_precision_recall_curves")
    plt.show()
else:
    no_samples_plot("Precision-Recall Curves", "11_precision_recall_curves")
chart_note("Clean and stress fake-detection tradeoff. Legend includes PR-AUC and sample count for each curve.")


## Optional Source-Level Analysis

Source-level plotting is intentionally skipped when the CSV only contains neutral or unassigned source values. The notebook does not infer source names from filenames or subgroups.


In [ ]:
source_values = sorted([
    s for s in scored.get("source", pd.Series(dtype=str)).dropna().astype(str).unique()
    if s and s.lower() not in {"unassigned", "unknown", "real", "fake"}
])
if len(source_values) < 2:
    print("Source-level charts skipped: source is not meaningful in this CSV.")
else:
    source_frame = scored[scored["source"].isin(source_values)].copy()
    rows = []
    for source, group in source_frame.groupby("source"):
        m = metrics_for(group, DEFAULT_THRESHOLD)
        rows.append({"source": source, "FPR": m["FPR"], "FNR": m["FNR"], "Accuracy": m["Accuracy"], "F1": m["F1 Score"], "count": m["Total samples"]})
    source_metrics = pd.DataFrame(rows).sort_values("FPR", ascending=False)
    display(source_metrics)
    source_metrics.to_csv(OUTPUT_DIR / "source_level_metrics.csv", index=False)
    print("Meaningful source categories detected; source-level metrics CSV saved. No source-wise dashboard chart is generated in this focused view.")


## Final Checks

This cell lists the saved analytical outputs. The benchmark CSV remains the source of truth; all charts and summaries are derived from it.


In [ ]:
print("Analysis outputs:")
for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file():
        print(path)
